In [2]:
import uproot
import pandas as pd
import numpy as np
import awkward as ak
import os
import json
import ROOT

In [3]:
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 100
import matplotlib.gridspec as gridspec
import matplotlib.patches as patches
import matplotlib.lines as lines
from matplotlib.collections import PatchCollection
plt.rcParams.update({'font.size': 15})

In [4]:
os.environ['ttH_yy_DIR'] = '/eos/user/e/elmazzeo/ttH@FCC-hh/results/2025-03-02'

In [5]:
basedir = os.path.join(os.environ.get('ttH_yy_DIR'), 'final')
outdir = os.path.join(os.environ.get('ttH_yy_DIR'), 'plots')
color_sequence = ["#3f90da","#ffa90e","#bd1f01","#94a4a2","#832db6","#a96b59","#e76300","#b9ac70","#717581","#92dadd"]
process = {
    'ttHyy' : { 'sample_list' : ['mgp8_pp_tth01j_5f_haa'], 
               'label' : r'$ttH \rightarrow \gamma\gamma$',
               'color' : color_sequence[0]}
}


selection = {
    'nocuts' : "All events"
}

process_infos = ["/eos/experiment/fcc/hh/utils/FCCDicts/FCChh_procDict_fcc_v07_II.json",
                 "/eos/experiment/fcc/hh/utils/FCCDicts/FCChh_procDict_fcc_v06_II.json"]

lumi = 3e7 # 3 * 10^7 pb-1 = 3 ab-1

In [6]:
def error_boxes (ax,x,y,x_err,y_err,fc='lightgrey',ec='None',alpha=0.5, hatch=None):

    # Create list for all the error patches
    errorboxes = []

    # Loop over data points; create box from errors at each point
    for xc,yc,xe,ye in zip(x,y,x_err.T,y_err.T):
        rect = patches.Rectangle((xc-xe,yc-ye),2*xe,2*ye)
        errorboxes.append(rect)

    # Create patch collection with specified colour/alpha
    pc = PatchCollection(errorboxes,facecolor=fc,alpha=alpha,edgecolor=ec, hatch=hatch)

    # Add collection to axes
    ax.add_collection(pc)

In [7]:
process_dicts = []
for inputname in process_infos :
    with open(inputname, 'r') as f :
        process_dicts.append(json.load(f))

In [8]:
from importlib import import_module
import sys
sys.path.insert(1, "../")

In [ ]:
# ok now I want to plot b-jets and photons variables
selection = {
    "nocuts" : "All events",
    "lep_channel" : "Photon and $b$-jet sel., $\geq$ 1 lepton",
    "pT_yy_bin1" : "0$\leq p_{T}(\gamma\gamma)$< 60 GeV",
    "pT_yy_bin2" : "60$\leq p_{T}(\gamma\gamma)$< 120 GeV",
    "pT_yy_bin3" : "120$\leq p_{T}(\gamma\gamma)$< 200 GeV",
    "pT_yy_bin4" : "200$\leq p_{T}(\gamma\gamma)$< 300 GeV",
    "pT_yy_bin5" : "$p_{T}(\gamma\gamma)\geq$ 300 GeV",
}
analysis_final = import_module("analysis_final")
variables_dict = analysis_final.histoList
variables_list = ['E_y1', 'pT_y1', 'eta_y1', 'true_E_y1', 'true_pT_y1', 'true_eta_y1', 'weight']

In [ ]:
df = {}
sow = {}

In [ ]:
for p in process.keys() :
    if p in list(df.keys()) :
        pass
    else :
        df[p] = {}
    print(p)
    variable_list1 = variables_list + ['pT_higgs'] if p=='ttHyy' else variables_list
    for s in selection.keys() :
        if s in list(df[p].keys()) :
            pass
        else :
            print("\t\t"+s)
            df1 = []
            sow[p] = []
            for sample in process[p]['sample_list'] :
                inputfile = os.path.join(basedir, sample+"_"+s+".root")
                # get sum of weights
                f = ROOT.TFile.Open(inputfile)
                sow[p].append(f.Get("SumOfWeights").GetVal())
                f.Close()
                # get sample dict
                for d in process_dicts :
                    if sample in list(d.keys()) :
                        process_dict = d.copy()
                        break
                # get sample
                with uproot.open(inputfile) as f :
                    df1.append(ak.to_dataframe(f['events'].arrays(expressions=variable_list1, library='ak')))
                    df1[-1]["weight"] = df1[-1]["weight"]/sow[p][-1]*process_dict[sample]['crossSection']*process_dict[sample]['kfactor']*process_dict[sample]['matchingEfficiency']
            df[p][s] = pd.concat(df1, copy=True, ignore_index=True)

In [ ]:
process_key = list(process.keys())[0]
process_label = process[process_key]['label']
sel_key = list(selection.keys())[0]
sel_label = selection[sel_label]

In [ ]:
ETA_BINS = np.array([0., 4.0, 6.0], dtype=np.float64)
ENERGY_BINS = np.array([20, 30, 40, 50, 60, 70, 80, 90, 100,
                       120, 140, 160, 180, 200,
                       220, 240, 270, 300,
                       350, 400, 450, 500, 600, 1000], dtype=np.float64)
x = 0.5*(ENERGY_BINS[:-1] + ENERGY_BINS[1:])
xerr = 0.5*(ENERGY_BINS[1:] - ENERGY_BINS[:-1])

In [ ]:
df1 = df[process_key][sel_key].copy()

In [ ]:
df1['eta_bin'] = np.digitize(np.abs(df1['true_eta_y1']), ETA_BINS)
df1['energy_bin'] = np.digitize(np.abs(df1['true_E_y1']), ENERGY_BINS)

In [ ]:
df1["idx"] = df1.index
df1 = df1.set_index(["eta_bin", "energy_bin", "idx"])

In [ ]:
for ibin in range(1, len(ETA_BINS)-1) :
    y = np.zeros(len(ENERGY_BINS)-1)
    for jbin in range(1, len(ENERGY_BINS)) :
        i_low = ETA_BINS[ibin-1]
        i_up = ETA_BINS[ibin]
        j_low = ENERGY_BINS[jbin-1]
        j_up = ENERGY_BINS[jbin] 
        
        df_all = df1.loc[ibin, jbin]
        observable = df_all['E_y1']/df_all['true_E_y1']-np.ones(len(df_all))
        resolution = np.std(observable)/(len(df_all)-1)
        y[jbin-1] = resolution
        # E / Etrue -1
        
        fig, ax = plt.subplots(
            nrows=1,
            ncols=1,
            figsize=(8,6)
        )
        ax.hist(observable, weights=df_all['weight']/df_all['weight'].sum(),
                bins=100, range=(-0.1,0.1), histtype='stepfilled', color=color_sequence[0], alpha=0.4)
        ax.axvline(0, ymin=0, ymax=0.7, color='darkgrey', ls='--', lw=2.)

        ax.text(0.05, 0.90, r"FCC-hh Simulation (Delphes)", transform=ax.transAxes, fontsize=16)
        ax.text(0.05, 0.82, process_label, transform=ax.transAxes, fontsize=16)
        ax.text(0.05, 0.74, str(i_low)+"$<|\eta^{truth}|<$"+str(i_up), transform=ax.transAxes, fontsize=15)
        ax.text(0.05, 0.66, str(j_low)+"$<E^{truth}<$"+str(j_up)+" GeV", transform=ax.transAxes, fontsize=15)
        xlabel = "Leading photon $E/E^{truth}-1$"
        ax.set_xlabel(xlabel)
        ax.set_ylabel(r'Normalised events')

        ymin,ymax = ax.get_ylim() 
        ax.set_ylim(ymin, ymax+np.abs((ymax-ymin))*0.65)
    
        plotname = "ereco_over_etrue_leading_photon_eta_bin_"+str(ibin)+"_energy_bin_"+str(jbin)
        plt.savefig(os.path.join(outdir, plotname+".pdf"), bbox_inches='tight')
        plt.savefig(os.path.join(outdir, plotname+".png"), bbox_inches='tight')
        plt.show()        

    fig, ax = plt.subplots(
        nrows=1,
        ncols=1,
        figsize=(8,6)
    )

    ax.errorbar(x, y, xerr=xerr, color=color_sequence[0], marker='o', lw=2, ls='none')

    ax.text(0.05, 0.90, r"FCC-hh Simulation (Delphes)", transform=ax.transAxes, fontsize=16)
    ax.text(0.05, 0.82, process_label, transform=ax.transAxes, fontsize=16)
    ax.text(0.05, 0.74, str(i_low)+"$<|\eta^{truth}|<$"+str(i_up), transform=ax.transAxes, fontsize=16)
    xlabel = "True photon energy [GeV]"
    ax.set_xlabel(xlabel)
    ax.set_ylabel(r'Std. dev. $E/E^{truth}-1$')

    ymin,ymax = ax.get_ylim() 
    ax.set_ylim(ymin, ymax+np.abs((ymax-ymin))*0.35)
    
    plotname = "ereco_over_etrue_leading_photon_eta_bin_"+str(ibin)
    plt.savefig(os.path.join(outdir, plotname+".pdf"), bbox_inches='tight')
    plt.savefig(os.path.join(outdir, plotname+".png"), bbox_inches='tight')
    plt.show()        